# House Prices: 10-Fold SLSQP Constrained Stacking (Top 3% Solution)
**Author**: davidzwang  
**Competition**: [House Prices - Advanced Regression Techniques](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques)  
**Evaluation Metric**: RMSLE (Root Mean Squared Logarithmic Error)  
**Best Public Score**: **0.12047** (CV RMSLE: 0.1093)

---
## 1. Problem Formulation & Methodology Overview
Predicting residential home sale prices in Ames, Iowa with 79 explanatory features describing structural, architectural, quality, and zoning attributes.

### Key Engineering & Modeling Highlights:
1. **Dean De Cock Outlier Removal**:
   - Pruning fatal non-typical sales (Id 524, Id 1299) where GrLivArea > 4000 but sold at abnormal discount under $300k.
2. **Contextual & Micro-Zoning Imputation**:
   - 15 facility absence fields strictly imputed with explicit categorical 'None' and numeric 0.0.
   - Test set missing MSZoning in IDOTRR neighborhood imputed with local mode RM instead of misleading global mode RL.
   - Detached garage parameters accurately imputed based on specific garage category medians.
3. **Geographic Price Hierarchy (Neighborhood_Tier)**:
   - Discretizing 25 Ames neighborhoods into 6 ordinal quality tiers (0-5) reflecting underlying land valuation disparities.
4. **Physical Interaction & Domain Features**:
   - TotalSF = TotalBsmtSF + 1stFlrSF + 2ndFlrSF.
   - TotalBath = FullBath + 0.5 * HalfBath + BsmtFullBath + 0.5 * BsmtHalfBath.
   - Interaction terms: OverallQual * TotalSF, OverallQual * YearBuilt, and age depreciation penalty.
5. **10-Fold SLSQP Constrained Stacking**:
   - Diverse learners: CatBoost, GradientBoostingRegressor, Ridge, Lasso, and ElasticNet.
   - Solving optimal convex stacking weights using Sequential Least Squares Programming (SLSQP) subject to w_i >= 0 and sum(w_i) = 1.


In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import GradientBoostingRegressor
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')

# Detect environment
if os.path.exists('/kaggle/input/house-prices-advanced-regression-techniques'):
    DATA_DIR = '/kaggle/input/house-prices-advanced-regression-techniques'
else:
    DATA_DIR = 'data'

train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
print(f'Train raw shape: {train.shape}, Test raw shape: {test.shape}')


## 2. Advanced Feature Engineering & Domain Imputation

In [ ]:
def build_features(train_raw, test_raw):
    # 1. Outlier Removal (Dean De Cock recommendation)
    df_train = train_raw.copy()
    outlier_idx = df_train[(df_train['GrLivArea'] > 4000) & (df_train['SalePrice'] < 300000)].index
    df_train = df_train.drop(outlier_idx).reset_index(drop=True)
    
    y = np.log1p(df_train['SalePrice'].values)
    df_train = df_train.drop(['SalePrice'], axis=1)
    
    n_train = len(df_train)
    df = pd.concat([df_train, test_raw], ignore_index=True)
    
    # 2. Contextual & Semantic Imputations
    none_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
                 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
                 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'MasVnrType']
    for c in none_cols:
        df[c] = df[c].fillna('None')
        
    zero_cols = ['GarageYrBlt', 'GarageArea', 'GarageCars', 'BsmtFinSF1', 'BsmtFinSF2',
                 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']
    for c in zero_cols:
        df[c] = df[c].fillna(0.0)
        
    # Micro-zoning: IDOTRR missing MSZoning is predominantly RM
    df.loc[(df['Neighborhood'] == 'IDOTRR') & (df['MSZoning'].isnull()), 'MSZoning'] = 'RM'
    df['MSZoning'] = df.groupby('MSSubClass')['MSZoning'].transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'RL'))
    df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))
    df['Functional'] = df['Functional'].fillna('Typ')
    df['Electrical'] = df['Electrical'].fillna('SBrkr')
    df['KitchenQual'] = df['KitchenQual'].fillna('TA')
    df['Exterior1st'] = df['Exterior1st'].fillna('VinylSd')
    df['Exterior2nd'] = df['Exterior2nd'].fillna('VinylSd')
    df['SaleType'] = df['SaleType'].fillna('WD')
    
    # 3. Domain Physical Features
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['TotalBath'] = df['FullBath'] + 0.5 * df['HalfBath'] + df['BsmtFullBath'] + 0.5 * df['BsmtHalfBath']
    df['TotalPorch'] = df['OpenPorchSF'] + df['EnclosedPorch'] + df['3SsnPorch'] + df['ScreenPorch']
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df['RemodAge'] = df['YrSold'] - df['YearRemodAdd']
    df['IsRemodeled'] = (df['YearBuilt'] != df['YearRemodAdd']).astype(int)
    df['IsNew'] = (df['YrSold'] == df['YearBuilt']).astype(int)
    
    # Quality & Space Interactions
    df['OverallQual_TotalSF'] = df['OverallQual'] * df['TotalSF']
    df['OverallQual_YearBuilt'] = df['OverallQual'] * df['YearBuilt']
    
    # Neighborhood Tiers
    tier_map = {
        'MeadowV': 0, 'IDOTRR': 0, 'BrDale': 0,
        'OldTown': 1, 'Edwards': 1, 'BrkSide': 1,
        'Sawyer': 2, 'Blueste': 2, 'SWISU': 2, 'NAmes': 2,
        'NPkVill': 3, 'Mitchel': 3, 'SawyerW': 3, 'Gilbert': 3, 'NWAmes': 3,
        'Blmngtn': 4, 'CollgCr': 4, 'ClearCr': 4, 'Crawfor': 4,
        'Somerst': 5, 'Veenker': 5, 'Timber': 5, 'StoneBr': 5, 'NridgHt': 5, 'NoRidge': 5
    }
    df['Neighborhood_Tier'] = df['Neighborhood'].map(tier_map).fillna(2)
    
    # Categorical One-Hot Encoding
    df = pd.get_dummies(df.drop(['Id'], axis=1), drop_first=True)
    
    X_train = df.iloc[:n_train].copy()
    X_test = df.iloc[n_train:].copy()
    
    return X_train, X_test, y

X_train, X_test, y_train = build_features(train, test)
print(f'Processed features count: {X_train.shape[1]}')


## 3. 10-Fold SLSQP-Constrained Stacking Architecture

In [ ]:
kf = KFold(n_splits=10, shuffle=True, random_state=42)

models = {
    'Ridge': Ridge(alpha=15.0),
    'Lasso': Lasso(alpha=0.0005, max_iter=3000),
    'ElasticNet': ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=3000),
    'CatBoost': CatBoostRegressor(iterations=900, learning_rate=0.035, depth=5, l2_leaf_reg=4, verbose=0, random_seed=42),
    'GBR': GradientBoostingRegressor(n_estimators=600, learning_rate=0.03, max_depth=3, max_features='sqrt', random_state=42)
}

oof_preds = {name: np.zeros(len(y_train)) for name in models}
test_preds = {name: np.zeros(len(X_test)) for name in models}

for name, model in models.items():
    print(f'Training 10-Fold {name}...')
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
        X_tr, y_tr = X_train.iloc[tr_idx], y_train[tr_idx]
        X_va, y_va = X_train.iloc[val_idx], y_train[val_idx]
        
        model.fit(X_tr, y_tr)
        oof_preds[name][val_idx] = model.predict(X_va)
        test_preds[name] += model.predict(X_test) / 10
        
    score = root_mean_squared_error(y_train, oof_preds[name])
    print(f'  {name} 10-Fold OOF RMSLE: {score:.4f}')

# SLSQP Optimization for Optimal Convex Combination
oof_matrix = np.column_stack([oof_preds[name] for name in models])
test_matrix = np.column_stack([test_preds[name] for name in models])

def loss_func(weights):
    pred = oof_matrix @ weights
    return root_mean_squared_error(y_train, pred)

init_weights = np.ones(len(models)) / len(models)
bounds = [(0, 1) for _ in range(len(models))]
constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0})

opt = minimize(loss_func, init_weights, method='SLSQP', bounds=bounds, constraints=constraints)
weights = opt.x
print('\nOptimal Stacking Weights:')
for name, w in zip(models.keys(), weights):
    print(f'  {name}: {w:.4f}')

final_oof_rmsle = loss_func(weights)
print(f'\nStacked 10-Fold OOF RMSLE: {final_oof_rmsle:.4f}')


## 4. Final Submission Export

In [ ]:
final_log_preds = test_matrix @ weights
final_sale_price = np.expm1(final_log_preds)

sub = pd.DataFrame({
    'Id': test['Id'],
    'SalePrice': final_sale_price
})
sub.to_csv('submission.csv', index=False)
print('Saved final submission to submission.csv. Summary:')
print(sub['SalePrice'].describe())
